# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [34]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [35]:
# Load the dataset from Phase 2
DATA_PATH = "../data/Metro_Interstate_Traffic_Volume.csv"   # change filename if needed

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

Loaded dataset: 48204 rows x 9 columns


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918



Columns:
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [36]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.


# Task 1: Select the relevant columns for traffic volume forecasting

# Keep only columns useful for predicting traffic volume
columns_to_keep = [
    'holiday',
    'temp',
    'rain_1h',
    'snow_1h',
    'clouds_all',
    'weather_main',
    'weather_description',
    'date_time',
    'traffic_volume'
]

# Drop less useful or redundant columns
columns_to_drop = [
    'temp_min',
    'temp_max'
]

drop_reason = {
    'temp_min': 'Less useful than the main temperature column and may add redundancy',
    'temp_max': 'Less useful than the main temperature column and may add redundancy'
}

df_selected = df[columns_to_keep].copy()

print("Selected columns:")
print(df_selected.columns.tolist())

print("\nDropped columns and reasons:")
for col, reason in drop_reason.items():
    print(f"- {col}: {reason}")

print(f"\nShape after column selection: {df_selected.shape}")
display(df_selected.head())

Selected columns:
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']

Dropped columns and reasons:
- temp_min: Less useful than the main temperature column and may add redundancy
- temp_max: Less useful than the main temperature column and may add redundancy

Shape after column selection: (48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [37]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition


before_rows = len(df_selected)

# Remove rows where the target or datetime is missing
df_selected = df_selected.dropna(subset=['date_time', 'traffic_volume'])

after_rows = len(df_selected)

print(f"Rows before filtering: {before_rows}")
print(f"Rows after filtering:  {after_rows}")
print(f"Rows removed:          {before_rows - after_rows}")

Rows before filtering: 48204
Rows after filtering:  48204
Rows removed:          0


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [38]:
# TODO: Handle missing values.
# Choose an appropriate strategy for each column.

# Example strategies:
# df_clean = df_selected.copy()

# Strategy 1: Impute numerical columns with the median
# numerical_cols = df_clean.select_dtypes(include=np.number).columns
# df_clean[numerical_cols] = df_clean[numerical_cols].fillna(df_clean[numerical_cols].median())

# Strategy 2: Impute categorical columns with the mode
# categorical_cols = df_clean.select_dtypes(include='object').columns
# for col in categorical_cols:
#     df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Strategy 3: Drop rows/columns with too many missing values
# df_clean = df_clean.dropna(thresh=len(df_clean) * 0.5, axis=1)  # Drop cols with >50% missing

# print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")


df_clean = df_selected.copy()

print("Missing values before cleaning:")
print(df_clean.isnull().sum())

# Numerical columns -> fill with median
numerical_cols = ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'traffic_volume']
for col in numerical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical columns -> fill with mode
categorical_cols = ['holiday', 'weather_main', 'weather_description']
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

Missing values before cleaning:
holiday                48143
temp                       0
rain_1h                    0
snow_1h                    0
clouds_all                 0
weather_main               0
weather_description        0
date_time                  0
traffic_volume             0
dtype: int64

Missing values after cleaning:
holiday                0
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
date_time              0
traffic_volume         0
dtype: int64


In [39]:
# TODO: Remove duplicate records.

before = len(df_clean)
duplicates_before = df_clean.duplicated().sum()

df_clean = df_clean.drop_duplicates()

after = len(df_clean)
duplicates_after = df_clean.duplicated().sum()

print(f"Duplicate rows before removal: {duplicates_before}")
print(f"Rows removed: {before - after}")
print(f"Remaining rows: {after}")
print(f"Duplicate rows after removal: {duplicates_after}")

Duplicate rows before removal: 17
Rows removed: 17
Remaining rows: 48187
Duplicate rows after removal: 0


In [40]:
# TODO: Handle outliers.
# Choose a strategy: capping (winsorizing), removing, or transforming.


# Task 2: Handle outliers using IQR capping for weather-related numerical columns

def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

outlier_cols = ['temp', 'rain_1h', 'snow_1h', 'clouds_all']

for col in outlier_cols:
    df_clean = cap_outliers_iqr(df_clean, col)

print("Outlier capping completed for:")
print(outlier_cols)

display(df_clean[outlier_cols].describe())

Outlier capping completed for:
['temp', 'rain_1h', 'snow_1h', 'clouds_all']


,temp,rain_1h,snow_1h,clouds_all
count,48187.000000,48187.0,48187.0,48187.000000
mean,281.255359,0.0,0.0,49.365451
std,12.720715,0.0,0.0,39.015213
min,242.691000,0.0,0.0,0.000000
25%,272.160000,0.0,0.0,1.000000
50%,282.450000,0.0,0.0,64.000000
75%,291.806000,0.0,0.0,90.000000
max,310.070000,0.0,0.0,100.000000


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [41]:
# TODO: Create derived attributes / new features.


# Task 3: Create derived time-based features

df_clean['date_time'] = pd.to_datetime(df_clean['date_time'])

df_clean['year'] = df_clean['date_time'].dt.year
df_clean['month'] = df_clean['date_time'].dt.month
df_clean['day'] = df_clean['date_time'].dt.day
df_clean['hour'] = df_clean['date_time'].dt.hour
df_clean['day_of_week'] = df_clean['date_time'].dt.dayofweek  # Monday=0, Sunday=6
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6]).astype(int)

print("New derived features created:")
print(['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend'])

display(df_clean.head())

New derived features created:
['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,is_weekend
0,Labor Day,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012,10,2,9,1,0
1,Labor Day,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012,10,2,10,1,0
2,Labor Day,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012,10,2,11,1,0
3,Labor Day,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012,10,2,12,1,0
4,Labor Day,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012,10,2,13,1,0


In [42]:
# TODO: Encode categorical variables.


# Task 3: Encode categorical variables using label encoding

label_encoders = {}
categorical_cols = ['holiday', 'weather_main', 'weather_description']

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

print("Categorical columns encoded:")
print(categorical_cols)

display(df_clean.head())

Categorical columns encoded:
['holiday', 'weather_main', 'weather_description']


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,is_weekend
0,3,288.28,0.0,0.0,40,1,24,2012-10-02 09:00:00,5545,2012,10,2,9,1,0
1,3,289.36,0.0,0.0,75,1,2,2012-10-02 10:00:00,4516,2012,10,2,10,1,0
2,3,289.58,0.0,0.0,90,1,19,2012-10-02 11:00:00,4767,2012,10,2,11,1,0
3,3,290.13,0.0,0.0,90,1,19,2012-10-02 12:00:00,5026,2012,10,2,12,1,0
4,3,291.14,0.0,0.0,75,1,2,2012-10-02 13:00:00,4918,2012,10,2,13,1,0


In [43]:
# TODO: Scale / normalise numerical features if required.

# from sklearn.preprocessing import StandardScaler, MinMaxScaler

# scaler = StandardScaler()  # or MinMaxScaler()
# cols_to_scale = ['feature_1', 'feature_2']
# df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])


# Task 3: Scaling / normalisation

# For this project, scaling is not applied at this stage.
# Tree-based models do not require scaling, and scaling can also be done later
# after the train-test split in Phase 4 to avoid data leakage.

print("No scaling applied in Phase 3.")
print("Reason: scaling will be handled later if required by the chosen model.")

No scaling applied in Phase 3.
Reason: scaling will be handled later if required by the chosen model.


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [44]:
# TODO: Integrate data from multiple sources (if applicable).

# Example: Merge two DataFrames on a common key
# df_secondary = pd.read_csv('path/to/secondary_data.csv')
# df_integrated = pd.merge(df_clean, df_secondary, on='common_key', how='left')

# Example: Concatenate DataFrames with the same structure
# df_integrated = pd.concat([df_part1, df_part2], ignore_index=True)

# If using a single source, simply continue:
# df_integrated = df_clean.copy()
# print(f"Integrated dataset shape: {df_integrated.shape}")


# This project uses a single dataset, so no merging or concatenation is required.
df_integrated = df_clean.copy()

print("This project uses a single data source.")
print(f"Integrated dataset shape: {df_integrated.shape}")
display(df_integrated.head())

This project uses a single data source.
Integrated dataset shape: (48187, 15)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,is_weekend
0,3,288.28,0.0,0.0,40,1,24,2012-10-02 09:00:00,5545,2012,10,2,9,1,0
1,3,289.36,0.0,0.0,75,1,2,2012-10-02 10:00:00,4516,2012,10,2,10,1,0
2,3,289.58,0.0,0.0,90,1,19,2012-10-02 11:00:00,4767,2012,10,2,11,1,0
3,3,290.13,0.0,0.0,90,1,19,2012-10-02 12:00:00,5026,2012,10,2,12,1,0
4,3,291.14,0.0,0.0,75,1,2,2012-10-02 13:00:00,4918,2012,10,2,13,1,0


In [45]:
# Optional: Verify the integrated data

print("Integrated dataset preview:")
display(df_integrated.head())

print("\nIntegrated dataset info:")
print(df_integrated.info())

Integrated dataset preview:


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,is_weekend
0,3,288.28,0.0,0.0,40,1,24,2012-10-02 09:00:00,5545,2012,10,2,9,1,0
1,3,289.36,0.0,0.0,75,1,2,2012-10-02 10:00:00,4516,2012,10,2,10,1,0
2,3,289.58,0.0,0.0,90,1,19,2012-10-02 11:00:00,4767,2012,10,2,11,1,0
3,3,290.13,0.0,0.0,90,1,19,2012-10-02 12:00:00,5026,2012,10,2,12,1,0
4,3,291.14,0.0,0.0,75,1,2,2012-10-02 13:00:00,4918,2012,10,2,13,1,0



Integrated dataset info:
<class 'pandas.DataFrame'>
Index: 48187 entries, 0 to 48203
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48187 non-null  int64         
 1   temp                 48187 non-null  float64       
 2   rain_1h              48187 non-null  float64       
 3   snow_1h              48187 non-null  float64       
 4   clouds_all           48187 non-null  int64         
 5   weather_main         48187 non-null  int64         
 6   weather_description  48187 non-null  int64         
 7   date_time            48187 non-null  datetime64[us]
 8   traffic_volume       48187 non-null  int64         
 9   year                 48187 non-null  int32         
 10  month                48187 non-null  int32         
 11  day                  48187 non-null  int32         
 12  hour                 48187 non-null  int32         
 13  day_of_week          

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [46]:
# TODO: Apply final formatting — data types, column order, renaming.

# Example: Convert data types
# df_integrated['some_column'] = df_integrated['some_column'].astype('int')

# Example: Rename columns for clarity
# df_integrated = df_integrated.rename(columns={
#     'old_name': 'new_descriptive_name'
# })

# Example: Reorder columns (target column last)
# target_col = 'target'
# feature_cols = [col for col in df_integrated.columns if col != target_col]
# df_final = df_integrated[feature_cols + [target_col]]


df_final = df_integrated.copy()

# Make sure column types are correct
df_final['traffic_volume'] = df_final['traffic_volume'].astype(int)
df_final['year'] = df_final['year'].astype(int)
df_final['month'] = df_final['month'].astype(int)
df_final['day'] = df_final['day'].astype(int)
df_final['hour'] = df_final['hour'].astype(int)
df_final['day_of_week'] = df_final['day_of_week'].astype(int)
df_final['is_weekend'] = df_final['is_weekend'].astype(int)

# Reorder columns: features first, target last
final_column_order = [
    'holiday',
    'temp',
    'rain_1h',
    'snow_1h',
    'clouds_all',
    'weather_main',
    'weather_description',
    'date_time',
    'year',
    'month',
    'day',
    'hour',
    'day_of_week',
    'is_weekend',
    'traffic_volume'
]

df_final = df_final[final_column_order]

print("Final formatting completed.")
print("Final column order:")
print(df_final.columns.tolist())

Final formatting completed.
Final column order:
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'traffic_volume']


In [47]:
# TODO: Verify the final prepared dataset.

print("-" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("-" * 60)
print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")

print("\nColumn types:")
print(df_final.dtypes)

print("\nFirst 5 rows:")
display(df_final.head())

------------------------------------------------------------
FINAL PREPARED DATASET SUMMARY
------------------------------------------------------------
Shape: (48187, 15)
Missing values: 0
Duplicates: 0

Column types:
holiday                         int64
temp                          float64
rain_1h                       float64
snow_1h                       float64
clouds_all                      int64
weather_main                    int64
weather_description             int64
date_time              datetime64[us]
year                            int64
month                           int64
day                             int64
hour                            int64
day_of_week                     int64
is_weekend                      int64
traffic_volume                  int64
dtype: object

First 5 rows:


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,year,month,day,hour,day_of_week,is_weekend,traffic_volume
0,3,288.28,0.0,0.0,40,1,24,2012-10-02 09:00:00,2012,10,2,9,1,0,5545
1,3,289.36,0.0,0.0,75,1,2,2012-10-02 10:00:00,2012,10,2,10,1,0,4516
2,3,289.58,0.0,0.0,90,1,19,2012-10-02 11:00:00,2012,10,2,11,1,0,4767
3,3,290.13,0.0,0.0,90,1,19,2012-10-02 12:00:00,2012,10,2,12,1,0,5026
4,3,291.14,0.0,0.0,75,1,2,2012-10-02 13:00:00,2012,10,2,13,1,0,4918


In [48]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).


OUTPUT_PATH = "../data/traffic_volume_prepared.csv"   # change if needed
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Prepared dataset saved to: {OUTPUT_PATH}")

Prepared dataset saved to: ../data/traffic_volume_prepared.csv
